<a href="https://colab.research.google.com/github/Rakshitanaik-27/EliteTech-DataScience/blob/main/Task3_End_To_End_WebApp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# Cell 1: Train ML Model and Save Scaler & Model
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

print("--- 1. DATASET LOADING & PREPROCESSING ---")
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(url, names=cols)

# Impute zero values with median
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_cols:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

# Train-Test Split
X = df.drop('Outcome', axis=1)
y = df['Outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Model Training
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Save Model & Scaler
with open('diabetes_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ Model (diabetes_model.pkl) and Scaler (scaler.pkl) saved successfully!")

--- 1. DATASET LOADING & PREPROCESSING ---
✅ Model (diabetes_model.pkl) and Scaler (scaler.pkl) saved successfully!


In [22]:
%%writefile app.py
import streamlit as st
import joblib
import numpy as np
import pandas as pd
import datetime
import plotly.express as px
import plotly.graph_objects as go

# ------------------------------------------------------------------------------
# PAGE CONFIGURATION
# ------------------------------------------------------------------------------
st.set_page_config(
    page_title="MediRisk AI - Diabetes Risk Analytics",
    page_icon="🩺",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS for Modern UI, Gradients, and Background Visual Polish
st.markdown("""
    <style>
    /* Main App Background Gradient */
    .stApp {
        background: linear-gradient(135deg, #f8fafc 0%, #eef2f6 100%);
    }

    /* Sleek Cards Styling */
    .metric-card {
        background: linear-gradient(145deg, #ffffff, #f1f5f9);
        padding: 18px;
        border-radius: 14px;
        box-shadow: 0 4px 12px rgba(0, 0, 0, 0.04);
        border: 1px solid #e2e8f0;
        text-align: center;
        transition: transform 0.2s ease;
    }
    .metric-title { color: #64748b; font-size: 14px; font-weight: 600; margin-bottom: 6px; }
    .metric-value { font-size: 26px; font-weight: 700; color: #0f172a; }

    .profile-box {
        background: #ffffff;
        padding: 22px;
        border-radius: 14px;
        border-left: 5px solid #3b82f6;
        box-shadow: 0 4px 12px rgba(0, 0, 0, 0.04);
    }

    /* Section Divider Styling */
    hr {
        border: 0;
        height: 1px;
        background: #cbd5e1;
        margin: 20px 0;
    }
    </style>
""", unsafe_allow_html=True)

# ------------------------------------------------------------------------------
# MODEL LOADING & SESSION STATE INITIALIZATION
# ------------------------------------------------------------------------------
@st.cache_resource
def load_model():
    return joblib.load('diabetes_model.pkl')

try:
    model = load_model()
except Exception as e:
    st.error("⚠️ Model file 'diabetes_model.pkl' not found. Ensure Cell 1 was executed to save the model.")
    st.stop()

# Initialize session state for prediction history
if 'history' not in st.session_state:
    st.session_state.history = pd.DataFrame(columns=["Date", "Patient Name", "Risk %", "Result", "Confidence"])

# ------------------------------------------------------------------------------
# SIDEBAR CONTROLS
# ------------------------------------------------------------------------------
st.sidebar.title("🩺 Control Panel")
st.sidebar.markdown("---")
st.sidebar.subheader("Patient Health Profile")

patient_name = st.sidebar.text_input("Patient Name", "Rakshita Naik")
gender = st.sidebar.selectbox("Gender", ["Female", "Male"])
age = st.sidebar.slider("Age (Years)", 21, 81, 33)
pregnancies = st.sidebar.slider("Pregnancies", 0, 17, 1)
glucose = st.sidebar.slider("Glucose Level (mg/dL)", 0, 200, 140)
blood_pressure = st.sidebar.slider("Blood Pressure (mm Hg)", 0, 122, 72)
skin_thickness = st.sidebar.slider("Skin Thickness (mm)", 0, 99, 22)
insulin = st.sidebar.slider("Insulin Level (mu U/ml)", 0, 846, 85)
bmi = st.sidebar.slider("BMI", 0.0, 67.1, 28.4)
pedigree = st.sidebar.slider("Diabetes Pedigree Function", 0.078, 2.42, 0.372)

st.sidebar.markdown("---")
analyze_btn = st.sidebar.button("🔍 Analyze & Save Record", type="primary", use_container_width=True)

# ------------------------------------------------------------------------------
# MACHINE LEARNING INFERENCE
# ------------------------------------------------------------------------------
input_data = np.array([[pregnancies, glucose, blood_pressure, skin_thickness, insulin, bmi, pedigree, age]])
prediction = model.predict(input_data)[0]
probabilities = model.predict_proba(input_data)[0]
risk_score = probabilities[1] * 100
confidence = max(probabilities) * 100
result_label = "High Risk" if prediction == 1 else "Low Risk"

# Save record when user clicks analyze button
if analyze_btn:
    today_str = datetime.date.today().strftime('%d %b %Y')
    new_record = pd.DataFrame([{
        "Date": today_str,
        "Patient Name": patient_name,
        "Risk %": f"{risk_score:.1f}%",
        "Result": result_label,
        "Confidence": f"{confidence:.1f}%"
    }])
    st.session_state.history = pd.concat([new_record, st.session_state.history], ignore_index=True)

# ------------------------------------------------------------------------------
# MAIN DASHBOARD LAYOUT
# ------------------------------------------------------------------------------
st.title("🩺 MediRisk AI — Diabetes Clinical Assessment")
st.caption("Interactive machine learning diagnostic tool with patient profile visualization and analytics.")

# ROW 1: TOP KPI METRIC CARDS
col1, col2, col3, col4 = st.columns(4)

with col1:
    st.markdown(f"""
        <div class="metric-card">
            <div class="metric-title">🩸 Glucose</div>
            <div class="metric-value">{glucose} <span style="font-size:14px; color:#64748b;">mg/dL</span></div>
        </div>
    """, unsafe_allow_html=True)

with col2:
    st.markdown(f"""
        <div class="metric-card">
            <div class="metric-title">🫀 Blood Pressure</div>
            <div class="metric-value">{blood_pressure} <span style="font-size:14px; color:#64748b;">mmHg</span></div>
        </div>
    """, unsafe_allow_html=True)

with col3:
    st.markdown(f"""
        <div class="metric-card">
            <div class="metric-title">⚖️ BMI Index</div>
            <div class="metric-value">{bmi:.1f}</div>
        </div>
    """, unsafe_allow_html=True)

with col4:
    st.markdown(f"""
        <div class="metric-card">
            <div class="metric-title">💉 Serum Insulin</div>
            <div class="metric-value">{insulin} <span style="font-size:14px; color:#64748b;">µU/mL</span></div>
        </div>
    """, unsafe_allow_html=True)

st.markdown("<br>", unsafe_allow_html=True)

# ROW 2: PREDICTION RESULT, RISK GAUGE & PATIENT PROFILE
res_col1, res_col2 = st.columns([1.6, 1.4])

with res_col1:
    st.subheader("🎯 Risk Evaluation & Diagnostic Output")

    p_col1, p_col2 = st.columns([1.1, 1.3])

    with p_col1:
        # Gauge Chart for Dynamic Risk Score Visualisation
        fig_gauge = go.Figure(go.Indicator(
            mode="gauge+number",
            value=risk_score,
            number={'suffix': "%", 'font': {'size': 24}},
            gauge={
                'axis': {'range': [0, 100], 'tickwidth': 1},
                'bar': {'color': "#ef4444" if prediction == 1 else "#10b981"},
                'steps': [
                    {'range': [0, 35], 'color': '#d1fae5'},
                    {'range': [35, 70], 'color': '#fef3c7'},
                    {'range': [70, 100], 'color': '#fee2e2'}
                ],
            }
        ))
        fig_gauge.update_layout(height=190, margin=dict(l=10, r=10, t=10, b=10), paper_bgcolor='rgba(0,0,0,0)')
        st.plotly_chart(fig_gauge, use_container_width=True)

    with p_col2:
        if prediction == 1:
            st.error("⚠️ **HIGH RISK DETECTED**")
            st.write("Patient parameters indicate elevated risk markers associated with Type-2 Diabetes.")
        else:
            st.success("✅ **LOW RISK / NORMAL**")
            st.write("Patient health indicators are within standard reference boundaries.")

        st.write(f"**Model Diagnostic Confidence:** `{confidence:.1f}%`")
        st.caption("Risk categories: Normal (<35%), Moderate (35%-70%), High (>70%).")

with res_col2:
    st.subheader("👤 Patient Profile Summary")
    st.markdown(f"""
        <div class="profile-box">
            <p style="margin-bottom:8px;"><strong>Patient Name:</strong> {patient_name}</p>
            <p style="margin-bottom:8px;"><strong>Demographics:</strong> {age} Yrs | {gender}</p>
            <p style="margin-bottom:8px;"><strong>BMI Category:</strong> {'Obese' if bmi >= 30 else 'Overweight' if bmi >= 25 else 'Normal'}</p>
            <p style="margin-bottom:0px;"><strong>Assessment Date:</strong> {datetime.date.today().strftime('%d %b %Y')}</p>
        </div>
    """, unsafe_allow_html=True)

st.markdown("<hr>", unsafe_allow_html=True)

# ROW 3: ADVANCED VISUALIZATIONS & ANALYTICS
tab1, tab2 = st.tabs(["📊 Comparative Patient Radar Chart", "📈 Model Feature Importances & Session Log"])

with tab1:
    st.markdown("#### Patient Biomarker Profile vs. Population Norms")

    # Normalize features for Radar Chart display against typical reference standards
    categories = ['Glucose', 'Blood Pressure', 'BMI', 'Insulin', 'Age', 'Pregnancies']
    patient_values = [
        min(glucose / 200 * 100, 100),
        min(blood_pressure / 120 * 100, 100),
        min(bmi / 50 * 100, 100),
        min(insulin / 250 * 100, 100),
        min(age / 80 * 100, 100),
        min(pregnancies / 15 * 100, 100)
    ]
    reference_values = [60, 60, 50, 35, 40, 10]

    fig_radar = go.Figure()

    fig_radar.add_trace(go.Scatterpolar(
        r=patient_values,
        theta=categories,
        fill='toself',
        name=f'{patient_name} (Current Input)',
        line_color='#3b82f6'
    ))
    fig_radar.add_trace(go.Scatterpolar(
        r=reference_values,
        theta=categories,
        fill='toself',
        name='Standard Normal Baseline',
        line_color='#10b981',
        opacity=0.4
    ))

    fig_radar.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
        showlegend=True,
        height=380,
        margin=dict(l=40, r=40, t=20, b=20),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)'
    )

    st.plotly_chart(fig_radar, use_container_width=True)

with tab2:
    hist_col1, hist_col2 = st.columns([1.2, 1.8])

    with hist_col1:
        st.markdown("#### Feature Importance")

        # Plotly Bar Chart for Feature Importances
        if hasattr(model, 'feature_importances_'):
            importances = model.feature_importances_
            features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'Pedigree', 'Age']
            df_feat = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values(by='Importance', ascending=True)

            fig_bar = px.bar(
                df_feat,
                x='Importance',
                y='Feature',
                orientation='h',
                color='Importance',
                color_continuous_scale='Viridis'
            )
            fig_bar.update_layout(
                height=320,
                showlegend=False,
                coloraxis_showscale=False,
                margin=dict(l=10, r=10, t=10, b=10),
                paper_bgcolor='rgba(0,0,0,0)',
                plot_bgcolor='rgba(0,0,0,0)'
            )
            st.plotly_chart(fig_bar, use_container_width=True)
        else:
            st.info("Feature importance display is not available for this model type.")

    with hist_col2:
        st.markdown("#### Session Prediction Records Log")
        if not st.session_state.history.empty:
            st.dataframe(st.session_state.history, use_container_width=True, height=280)
        else:
            st.info("No records logged in the current session. Adjust the clinical sliders and click **Analyze & Save Record**.")

Overwriting app.py


In [23]:
# Cell 3: Dependencies, Launch Streamlit & Generate Live Cloudflare URL
import subprocess
import re
import time

# 1. Install necessary dependencies
!pip install streamlit plotly -q
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

# 2. Terminate any existing background processes
!pkill streamlit
!pkill cloudflared

# 3. Launch Streamlit in background
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.enableCORS=false",
    "--server.enableXsrfProtection=false",
    "--server.headless=true",
    "--server.port=8501"
])

print("⏳ Starting Medical Streamlit Engine...")
time.sleep(5)

# 4. Launch Cloudflare Tunnel to expose local port 8501
process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print("⏳ Generating secure dashboard link...")
time.sleep(3)

# 5. Extract and display the live web application URL
for line in process.stderr:
    if "trycloudflare.com" in line:
        url_match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if url_match:
            live_url = url_match.group(0)
            print("\n" + "="*60)
            print(f"🩺 YOUR LIVE MEDICAL DASHBOARD URL: {live_url}")
            print("="*60 + "\n")
            print("Click the link above to open your interactive health app!")
            break

cloudflared: Text file busy
⏳ Starting Medical Streamlit Engine...
⏳ Generating secure dashboard link...

🩺 YOUR LIVE MEDICAL DASHBOARD URL: https://imperial-anonymous-changelog-mel.trycloudflare.com

Click the link above to open your interactive health app!


In [19]:
!pkill streamlit
!pkill cloudflared